# Lesson 9.2: Follow a tool request around a bounded loop

An investigation might look up an order, then decide whether the result is enough for a draft. The scripted responder below gives the same lookup request each time it sees the opening question. Follow the request, the access check and the record that returns to the loop before the final draft appears.

[Companion lesson](../modules/09-agent-loops/02-code-workshop.md) · [Original-source walkthrough](../modules/09-agent-loops/05-source-walkthrough.md)

Run one cell at a time from top to bottom. The programme uses the fictional records shown in the code. No API key or model download is needed.

## Predict the result

Follow the first proposal into the tool result and then into the final draft. Which value tells you how the loop stopped?

In [ ]:
ORDERS = {"O-17": {"ordered": 12, "reported_received": 10}}

def lookup_order(order_id):
    return dict(ORDERS[order_id])

TOOLS = {"lookup_order": lookup_order}

def scripted_model(messages):
    if messages[-1]["role"] == "tool":
        record = messages[-1]["result"]
        difference = record["ordered"] - record["reported_received"]
        return {"kind": "final", "text": f"Reported difference: {difference}"}
    return {
        "kind": "tool", "name": "lookup_order",
        "arguments": {"order_id": "O-17"}, "call_id": "call-1"
    }

def investigate(question, choose, permitted_ids, max_steps=3):
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        proposal = choose(messages)
        if proposal["kind"] == "final":
            return {"status": "draft", "text": proposal["text"], "trace": messages}
        if proposal["kind"] != "tool" or proposal["name"] not in TOOLS:
            raise ValueError("Unsupported action")
        arguments = proposal["arguments"]
        if set(arguments) != {"order_id"}:
            raise ValueError("Unexpected argument fields")
        order_id = arguments["order_id"]
        if not isinstance(order_id, str) or order_id not in permitted_ids:
            raise PermissionError("Order is outside the permitted scope")
        result = TOOLS[proposal["name"]](order_id=order_id)
        messages.append({"role": "assistant", "proposal": proposal})
        messages.append({"role": "tool", "call_id": proposal["call_id"], "result": result})
    return {"status": "needs_review", "reason": "Step budget exhausted", "trace": messages}

answer = investigate("Investigate O-17", scripted_model, {"O-17"})
print(answer["status"], answer["text"])

## Inspect the result

`print` displays values below the cell. When present, `assert` checks a specific expectation and raises an error if it fails.

In [ ]:
print(answer["trace"])
assert answer["status"] == "draft"

## Try a change

Allow just one step for the same question. Can the loop return the final draft after the lookup?

Make a prediction before opening the worked answer.

In [ ]:
# Try the change here, or make a temporary edit in the first code cell.

<details>
<summary>Worked answer</summary>

The lookup occurs, but the loop reaches its limit before the scripted responder can use the result to write the draft.

```python
short_run = investigate("Investigate O-17", scripted_model, {"O-17"}, max_steps=1)
assert short_run["status"] == "needs_review"
```

</details>